In [1]:
#Apply Raw
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time

# Load dataset
df= pd.read_csv("data/AmesHousing_engineered.csv")

# Drop target and ID columns
X_raw = df.drop(columns=["SalePrice", "PID", "Order"], errors="ignore")
print("Features shape (raw version):", X_raw.shape)

#Define Cluster Parameters
k_values = range(2, 9)  # clusters 2–8 for KMeans, GMM, Agglomerative, Spectral
n_init = 10  # random initialization for KMeans, GMM, Spectral
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch



Features shape (raw version): (2930, 172)


In [19]:
#K-Means
start_time = time.time()
kmeans_raw = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    kmeans.fit(X_raw)
    labels = kmeans.labels_
    sil, db, ch = compute_metrics(X_raw, labels) #K-Means on Raw Featureslabels)
    kmeans_raw.append({"algorithm":"K-Means","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")

Runtime: 2.4888126850128174 seconds
K-Means runtime: 2.4888 seconds


In [20]:
#GMM on Raw Features
start_time = time.time()
gmm_raw = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    gmm.fit(X_raw)
    labels = gmm.predict(X_raw)
    sil, db, ch = compute_metrics(X_raw, labels)
    gmm_raw.append({"algorithm":"GMM","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")

Runtime: 234.54314374923706 seconds
GMM runtime: 234.5431 seconds


In [21]:
#Agglomerative Clustering
start_time = time.time()
agg_raw = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    agg.fit(X_raw)
    labels = agg.labels_
    sil, db, ch = compute_metrics(X_raw, labels)
    agg_raw.append({"algorithm":"Agglomerative","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")

Runtime: 5.227984666824341 seconds
Agglomerative runtime: 5.2280 seconds


In [22]:
#Spectral Clustering
start_time = time.time()
spectral_raw = []
for k in k_values:
    spectral = SpectralClustering(n_clusters=k, affinity='nearest_neighbors', n_init=n_init, random_state=42)
    spectral.fit(X_raw)
    labels = spectral.labels_
    sil, db, ch = compute_metrics(X_raw, labels)
    spectral_raw.append({"algorithm":"Spectral","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")

Runtime: 7.129442453384399 seconds
Spectral runtime: 7.1294 seconds


In [23]:
#DBSCAN
start_time = time.time()
dbscan_raw = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    dbscan.fit(X_raw)
    labels = dbscan.labels_
    sil, db, ch = compute_metrics(X_raw, labels)
    dbscan_raw.append({"algorithm":"DBSCAN","preprocessing":"raw","eps":eps,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds") 

Runtime: 1.191298246383667 seconds
DBSCAN runtime: 1.1913 seconds


In [24]:
start_time = time.time()
from sklearn.cluster import Birch

birch_raw = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X_raw)

    if len(set(labels)) > 1:
        sil, db, ch = compute_metrics(X_raw, labels)
        birch_raw.append({
            "algorithm": "BIRCH",
            "preprocessing": "raw",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Birch runtime: {runtime:.4f} seconds")

Runtime: 4.779778003692627 seconds
Birch runtime: 4.7798 seconds


In [25]:
start_time = time.time()
from sklearn.cluster import OPTICS

optics_raw = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_raw)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_raw, labels)
        optics_raw.append({
            "algorithm": "OPTICS",
            "preprocessing": "raw",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")

c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


Runtime: 58.015570878982544 seconds
Optics runtime: 58.0156 seconds


In [26]:
import csv

ames_results_raw = (kmeans_raw + gmm_raw + agg_raw + spectral_raw + dbscan_raw + birch_raw + optics_raw)

# Desired column order
keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/ames_data/ames_raw.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(ames_results_raw)